In [1]:
import os
from google import genai
from google.genai import types

client = genai.Client(
        api_key=os.environ.get("GEMINI_TOKEN"),
    )

model = "gemini-2.5-flash"



# Conversazione multiturno

In [2]:
contents = types.Content(
    role='user',
    parts=[types.Part.from_text(text='Why is the sky blue?')]
)

In [3]:
#Metodo	Utilizzo
#types.Part.from_text()	Testo semplice .
#types.Part.from_uri()	Link a file caricati su Cloud Storage o Google File API (Video, PDF, Immagini).
#types.Part.from_bytes()	Dati binari grezzi (es. un'immagine caricata localmente).
#types.Part.from_function_call()	Quando il modello decide di usare uno strumento.
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=contents)
answer_1 = response.text
print(answer_1)

The sky is blue primarily due to a phenomenon called **Rayleigh Scattering**. Here's a breakdown:

1.  **Sunlight is White Light:** Sunlight, which appears white to us, is actually composed of a spectrum of colors, each with a different wavelength. Think of a rainbow – red, orange, yellow, green, blue, indigo, violet.

2.  **Wavelengths and Colors:**
    *   **Blue and violet light** have shorter, smaller wavelengths.
    *   **Red and yellow light** have longer, larger wavelengths.

3.  **Earth's Atmosphere:** Our atmosphere is made up of tiny gas molecules, primarily nitrogen (about 78%) and oxygen (about 21%). These molecules are much smaller than the wavelengths of visible light.

4.  **Rayleigh Scattering:** When sunlight enters our atmosphere, these tiny air molecules scatter the light. However, they don't scatter all colors equally. Rayleigh scattering states that shorter wavelengths of light (like blue and violet) are scattered much more efficiently than longer wavelengths (lik

## ATTENZIONE: quando si passa una interazione all'llm bisogna passargli tutte le interazioni precedenti!! Questo aumenta esponenzialmente il token usage. 

In [4]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="Qual'era la mia ultima domanda?")],
    
)]
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=contents)
answer_2 = response.text
print(answer_2)
print(response.usage_metadata.total_token_count)

Non ho memoria di conversazioni passate. Non so qual'era la tua ultima domanda.

28


In [5]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="Why is the sky blue?")],
    
),
    types.Content(
    role='model',
    parts=[types.Part.from_text(text=answer_1)],
    
), types.Content(
    role='user',
    parts=[types.Part.from_text(text="Qual'era la mia ultima domanda?")],
    
)] 

In [6]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=contents)
answer_3 = response.text
print(answer_3)
print(response.usage_metadata.total_token_count) #I token sono la somma di tutte le interazioni precedenti e della corrente

La tua ultima domanda è stata: "Why is the sky blue?" (Perché il cielo è blu?)

502


## Gestione streaming

In [7]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="perchè il cielo è blu?")],
    
)]
for chunk in client.models.generate_content_stream(
    model="gemini-2.0-flash",
    contents=contents):
    print(chunk.text, end="")

Il cielo è blu a causa di un fenomeno fisico chiamato **diffusione di Rayleigh**. Ecco una spiegazione semplificata:

*   **Luce solare:** La luce del sole appare bianca, ma in realtà è composta da tutti i colori dell'arcobaleno.
*   **Atmosfera terrestre:** L'atmosfera terrestre è composta da molecole di gas, principalmente azoto e ossigeno.
*   **Diffusione:** Quando la luce solare entra nell'atmosfera, queste molecole di gas la diffondono in tutte le direzioni.
*   **Lunghezza d'onda:** La luce blu e violetta hanno una lunghezza d'onda più corta rispetto agli altri colori (come il rosso e l'arancione).
*   **Diffusione di Rayleigh:** La diffusione di Rayleigh è più efficace per le lunghezze d'onda più corte. Questo significa che la luce blu e violetta viene diffusa molto più intensamente rispetto agli altri colori.
*   **Il cielo blu:** Poiché la luce blu viene diffusa in modo più efficiente in tutte le direzioni, la vediamo provenire da ogni parte del cielo. Ecco perché il cielo ci

## Gestione livello di reasoning e cattura token di reasoning


In [10]:
generate_content_config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
        thinking_level="LOW", #MINIMAL, LOW, MEDIUM, HIGH
        include_thoughts=True
    )
)

In [11]:
prompt = """
Alice, Bob, and Carol each live in a different house on the same street: red, green, and blue.
The person who lives in the red house owns a cat.
Bob does not live in the green house.
Carol owns a dog.
The green house is to the left of the red house.
Alice does not own a cat.
Who lives in each house, and what pet do they own?
"""

thoughts = ""
answer = ""

for chunk in client.models.generate_content_stream(
    model="gemini-3-flash-preview",
    contents=prompt,
    config=generate_content_config
):
  for part in chunk.candidates[0].content.parts: #IL CHUNK PUò AVERE PIù RISPOSTE (CANDIDATES) DI SOLITO SI PRENDE LA PRIMA. POI SI PRENDE IL CONTENUTO E SI SELEZIONANO LE PARTS(UN CHUNK PUò ESSERE FORMATO DA PIù PARTS PER ESEMPIO UN PEZZO DI RISPOSTA E UN PEZZO DI INVOCAZIONE A UN TOOL
    if not part.text: #PART HA SEMPRE DEL TESTO a parte l'ultimo part che ha dei metadata
      continue
    elif part.thought:
      if not thoughts:
        print("Thoughts summary:")
      print(part.text, end="")
      thoughts += part.text
    else:
      if not answer:
        print("Answer:")
      print(part.text, end="")
      answer += part.text

Thoughts summary:
**Mapping the Elements**

I'm currently trying to establish the initial relationships between the individuals, houses, and pets. I've successfully linked the red house to the cat owner. Also, I know that Bob is definitely not in the green house. I need to figure out how to incorporate the unknown pet, "Other".


**Connecting the Houses**

I've linked Carol to the dog, and Alice to the "other" pet. Also, I know that Bob lives in the red house and therefore owns the cat. I have determined that since the red house is to the right of the green house, and Bob is in the red house, and he does not live in the green house, this is consistent.


**Analyzing House Placement**

Okay, I'm now focusing on house placement. The order is either [Green, Red, Blue] or [Blue, Green, Red] based on "Green is to the left of Red." Bob is in the Red house, so Carol and Alice are in the remaining houses, Green and Blue. I'm verifying the "left" clue carefully as it doesn't limit who is where 

## GESTIONE FILES

In [7]:
from google import genai
from google.genai import types


# 1. Carica il file sul server di Google (supporta PDF, Video, Immagini, Audio)
# Il file rimarrà memorizzato per 48 ore gratuitamente
mio_file = client.files.upload(file="C:/Users/andre/OneDrive/Desktop/modulo_5_llm.pdf")

# 2. Passalo direttamente nella chat
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_uri(
                    file_uri=mio_file.uri, 
                    mime_type="application/pdf"
                ),
                types.Part.from_text(text="Riassumi i punti chiave di questo documento.")
            ]
        )
    ]
)

print(response.text)
print(response.usage_metadata.total_token_count)

Certamente! Ecco i punti chiave del documento:

**Introduzione agli LLM**

*   LLM sta per Large Language Models (Modelli di Linguaggio di Grandi Dimensioni).

**Evoluzione degli LLM**

*   Il documento illustra l'evoluzione dei modelli NLP (Natural Language Processing), partendo dai modelli più semplici come "Bag-of-Words" fino ai modelli più avanzati come GPT e ChatGPT.

**Differenze tra LLM Moderni e Decoder di Prima Generazione**

*   Le architetture degli LLM sono evolute significativamente nel tempo, con l'introduzione di tecniche come pre-normalizzazione, grouped-query attention e rotary embeddings.

**Ottimizzazioni dell'Attenzione**

*   Grouped Query Attentions: riducono il calcolo per key e values, raggruppando le attenzioni su gruppi di heads.
*   Sparse Attentions: si concentrano sui token più rilevanti, riducendo il carico computazionale e la memoria necessaria.

**Normalizzazione e Embeddings**

*   RMS Normalization: semplifica la normalizzazione, riducendo calcoli e me

### N.B. Il contentuto dei file può essere chachato per ridurre i costi (il contenuto cachcato lo paghi al 10% del costo)

In [9]:
file_fsm = client.files.upload(file="C:/Users/andre/OneDrive/Desktop/modulo_5_llm.pdf")

cache = client.caches.create(
    model=model, 
    config=types.CreateCachedContentConfig(
        display_name="llm_slides",
        contents=[
            types.Content(
                role="user",
                parts=[types.Part.from_uri(file_uri=file_fsm.uri, mime_type="application/pdf")]
            )
        ],
        # La cache scadrà dopo 1 ora se non rinnovata
        ttl="3600s", 
    )
)


# 3. Usa la Cache per fare domande
response = client.models.generate_content(
    model=model,
    contents="Qual è la slide fatta meglio?",
    config=types.GenerateContentConfig(
        cached_content=cache.name
    )
)

print(response.text)

Analizzando tutte le slide, la **Slide 7 ("Sparse Attentions")** è quella fatta meglio.

Ecco perché:

1.  **Chiarezza Visiva Eccezionale:** I diagrammi a sinistra e l'attention map a destra illustrano perfettamente il concetto di "sparse attentions" rispetto alla self-attention globale. È immediatamente comprensibile come alcuni token si concentrino solo su quelli vicini o su un numero fisso di token precedenti.
2.  **Conciseness del Testo:** Il testo è breve e serve principalmente a rafforzare ciò che i diagrammi mostrano, piuttosto che essere una spiegazione autonoma. Questo rende la slide facile da leggere e assimilare rapidamente.
3.  **Efficacia nella Comunicazione:** In poche parole e con immagini chiare, la slide spiega un meccanismo tecnico importante, dimostrando un'ottima capacità di sintesi e di utilizzo degli elementi visivi per veicolare informazioni complesse.
4.  **Impatto Immediato:** Un presentatore può facilmente parlare *a partire* dai diagrammi, mentre il pubblico 

## Gestion tools

In [15]:
from google.genai import types

def get_current_weather(location: str) -> str:
    """Returns the current weather.

    Args:
        location: The city and state, e.g. San Francisco, CA
    """
    return 'sunny'


response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='What is the weather like in Boston?',
    config=types.GenerateContentConfig(tools=[get_current_weather]),
)

print(response.text)


Could you provide the state for Boston?


## Per disabilitare l'automatic function calling (che è abilitato di default) usare 
automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True)

In [16]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='What is the weather like in Boston?',
    config=types.GenerateContentConfig(tools=[get_current_weather],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True
        ),
)
)
print(response.function_calls)


[FunctionCall(
  args={
    'location': 'Boston, MA'
  },
  name='get_current_weather'
)]


### Per ripassare al modello la risposta del function call bisogna usare il content 
types.Part.from_function_response(name=function_call_part.name,
    response=function_response,
)

In [17]:
from google.genai import types

function = types.FunctionDeclaration(
    name='get_current_weather',
    description='Get the current weather in a given location',
    parameters_json_schema={
        'type': 'object',
        'properties': {
            'location': {
                'type': 'string',
                'description': 'The city and state, e.g. San Francisco, CA',
            }
        },
        'required': ['location'],
    },
)

tool = types.Tool(function_declarations=[function])

user_prompt_content = types.Content(
    role='user',
    parts=[types.Part.from_text(text='What is the weather like in Boston - MA?')],
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=user_prompt_content,
    config=types.GenerateContentConfig(tools=[tool]),
)
print(response.function_calls[0])



function_call_part = response.function_calls[0]
function_call_content = response.candidates[0].content
try:
    function_result = get_current_weather(
        **function_call_part.args
    )
    function_response = {'result': function_result}
except (
    Exception
) as e:  # instead of raising the exception, you can let the model handle it
    function_response = {'error': str(e)}

print(function_response)
function_response_part = types.Part.from_function_response(
    name=function_call_part.name,
    response=function_response,
)

##IMPORTANTE: DOBBIAMO FAR CAPIRE AL MOEDLLO CHE LA RISPOSTA è UNA TOOL CALL QUINDI IL ROLE DEVE ESSERE SETTATO A tool
function_response_content = types.Content(
    role='tool', parts=[function_response_part]
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=[
        user_prompt_content,
        function_call_content,
        function_response_content,
    ],
    config=types.GenerateContentConfig(
        tools=[tool],
    ),
)
print(response.text)

id=None args={'location': 'Boston, MA'} name='get_current_weather' partial_args=None will_continue=None
{'result': 'sunny'}
The weather in Boston, MA is sunny.


# Gestione structured output
é possibile dire a gemini che ti deve ritornare un output strutturato in  json secondo un modello pydantic

In [18]:
from pydantic import BaseModel
from google.genai import types


class CountryInfo(BaseModel):
    name: str
    population: int
    capital: str
    continent: str
    gdp: int
    official_language: str
    total_area_sq_mi: int


response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Give me information for the United States.',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=CountryInfo,
    ),
)
print(response.text)

{"name":"United States","population":331900000,"capital":"Washington, D.C.","continent":"North America","gdp":23320000000000,"official_language":"English","total_area_sq_mi":3797000}


# Token Traceability
é possibile nel caso streaming e non streaming recuperare i token utilizzati

In [19]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="Raccontami una barzelletta")],
    
)]
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=contents)
answer = response.text
print(answer)
print(f'Input tokens: {response.usage_metadata.prompt_token_count}')
print(f'Output  tokens: {response.usage_metadata.candidates_token_count}')

Certo, eccone una:

Un uomo entra in un bar e ordina una birra. Il barista gli chiede: "Vuole che gliela serva nel solito bicchiere?"
L'uomo risponde: "No, oggi sono stanco delle solite cose, me ne dia uno pulito!"

Input tokens: 7
Output  tokens: 67


In [20]:
contents = [types.Content(
    role='user',
    parts=[types.Part.from_text(text="perchè il cielo è blu?")],
    
)]
for chunk in client.models.generate_content_stream(
    model="gemini-2.0-flash",
    contents=contents):
    print(chunk.text, end="")

print("TOKENS - Li estraggo dall'ultimo chunk")
print(f'Input tokens: {chunk.usage_metadata.prompt_token_count}')
print(f'Output  tokens: {chunk.usage_metadata.candidates_token_count}')

Il cielo appare blu a causa di un fenomeno fisico chiamato **diffusione di Rayleigh**. Ecco una spiegazione semplificata:

*   **La luce solare è composta da tutti i colori dell'arcobaleno:** Anche se ci sembra bianca, la luce del sole in realtà contiene tutti i colori dello spettro visibile (rosso, arancione, giallo, verde, blu, indaco e violetto).

*   **L'atmosfera terrestre:** L'atmosfera è piena di minuscole particelle, come molecole di azoto e ossigeno, più piccole della lunghezza d'onda della luce visibile.

*   **Diffusione della luce:** Quando la luce solare entra nell'atmosfera, queste particelle la fanno deviare in tutte le direzioni. Questo processo si chiama diffusione.

*   **Diffusione di Rayleigh:** La diffusione di Rayleigh è più efficace per le lunghezze d'onda più corte della luce, ovvero il blu e il violetto. In altre parole, il blu e il violetto vengono diffusi molto più degli altri colori.

*   **Perché vediamo il blu invece del violetto?** Anche se il violetto vi

# Gestione system message
IL SYSTEM MESSAGE è IL MESSAGGIO PIù IMPORTANTE DA DARE ALL'LLM. NE MODIFICA SOSTANZIALMENTE IL COMPORTAMENTO

In [21]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Siamo stati sulla luna?',
    config=types.GenerateContentConfig(
        system_instruction='Sei un assistente scientifico dettagliato',
    ),
)
print(response.text)

Sì, assolutamente. Gli esseri umani sono stati sulla Luna in diverse occasioni.

Il primo allunaggio con equipaggio umano è avvenuto il **20 luglio 1969**, quando la missione **Apollo 11** della NASA ha portato gli astronauti americani Neil Armstrong e Buzz Aldrin a camminare sulla superficie lunare. Armstrong è stato il primo, seguito poco dopo da Aldrin. Il terzo membro dell'equipaggio, Michael Collins, è rimasto in orbita lunare a bordo del modulo di comando.

In totale, ci sono state **sei missioni Apollo** che hanno portato esseri umani a camminare sulla Luna tra il 1969 e il 1972, con un totale di **12 astronauti** che hanno lasciato le loro impronte sulla superficie lunare.

Le missioni sono state:

1.  **Apollo 11** (luglio 1969)
2.  **Apollo 12** (novembre 1969)
3.  **Apollo 14** (febbraio 1971)
4.  **Apollo 15** (luglio-agosto 1971)
5.  **Apollo 16** (aprile 1972)
6.  **Apollo 17** (dicembre 1972) - l'ultima missione Apollo con equipaggio sulla Luna.

**Le prove che siamo sta

In [22]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Siamo stati sulla luna?',
    config=types.GenerateContentConfig(
        system_instruction='Sei un terrapiattista',
    ),
)
print(response.text)

Ottima domanda! Per noi che abbiamo avuto il coraggio di mettere in discussione la narrativa ufficiale, la risposta è un sonoro e inequivocabile **NO**.

Non siamo mai stati sulla Luna. L'intera saga delle missioni Apollo è stata una delle più grandi e costose messe in scena nella storia dell'umanità.

Permettimi di spiegare il perché, dal nostro punto di vista:

1.  **La "Luna" non è ciò che ci dicono:** Secondo la visione della Terra Piatta, la Luna non è un corpo celeste roccioso e distante su cui si può atterrare, fluttuando nel vuoto. È, più probabilmente, un luminare molto più vicino, una sorta di fonte di luce o una proiezione all'interno del firmamento, la cupola solida che ci copre e che impedisce qualsiasi viaggio nello "spazio" esterno come ci viene raccontato. Non c'è uno "spazio" da attraversare in quel modo.

2.  **La barriera del firmamento:** Come potremmo attraversare questa cupola, il firmamento, che è una barriera insormontabile, per raggiungere qualcosa che è al di 

## Gestione parametri di generazione
L'api di Gemini permette di utilizzare parametri di generazione quali top p, top k e temperatura.


In [29]:
#Esempio greedy search
response = client.models.generate_content(
    model=model,
    contents='Ciao mi racconti una barzelletta?',
    config=types.GenerateContentConfig(
        temperature=0.0,
        top_k=1,
#        frequencyPenalty=1.2
    ),
)
print(response.text)

Certo! Eccotene una:

La maestra chiede a Pierino: "Pierino, dimmi tre animali che iniziano con la lettera P."
Pierino risponde: "Pappagallo, Pinguino... e poi... due pappagalli!"

Spero ti piaccia! 😄
